# ROI locator and GLWD v2

Set `ROI_NAME` to a name in `rois.json` (for example `Rio Negro basin`). Every ROI gets a location panel and a zoomed GLWD v2 class map. **Belém** also gets the middle panel, because that is the site with the São José do Ribamar tide gauge.

GLWD is `GLWD_v2_0_main_class.tif` (EPSG:4326, 15 arc-seconds). Names are from `GLWD_Legend_v2_0.csv`. Dryland, nodata, and classes under 1% of the wetland pixels in the bbox are left off the map. Basemap: OpenStreetMap (`© OpenStreetMap contributors`).


In [ ]:
import csv
import json
import math
import os
import sys
import unicodedata
import urllib.request
from pathlib import Path

# This shell exports PROJ_DATA from a different conda install. Rasterio in
# hydropol needs that env's proj.db, set before rasterio is imported.
_proj = Path(sys.executable).resolve().parents[1] / "share" / "proj"
if (_proj / "proj.db").exists():
    os.environ["PROJ_DATA"] = str(_proj)
    os.environ["PROJ_LIB"] = str(_proj)

import matplotlib.pyplot as plt
import numpy as np
import rasterio
from matplotlib.patches import Patch, Polygon as MplPolygon
from matplotlib.ticker import FixedLocator, FuncFormatter
from PIL import Image
from rasterio.crs import CRS
from rasterio.enums import Resampling
from rasterio.transform import from_bounds as transform_from_bounds
from rasterio.warp import reproject
from rasterio.windows import from_bounds as window_from_bounds

HERE = Path.cwd()
if not (HERE / "osm_cache").exists() and (HERE / "topics" / "11-B-FLOODED-FOREST" / "georg_dev").exists():
    HERE = HERE / "topics" / "11-B-FLOODED-FOREST" / "georg_dev"

GLWD_ROOT = Path("/home/georg/data/Biomass_hackathon/GLWD_v2_0_combined_classes_tif")
ROIS_PATH = HERE.parent / "rois.json"

# Name from rois.json. "Belém" also draws the Ribamar tide-gauge panel.
ROI_NAME = "Aranapu"


def fold(text):
    text = unicodedata.normalize("NFKD", text)
    return "".join(c for c in text if not unicodedata.combining(c)).casefold().strip()


rois = json.loads(ROIS_PATH.read_text(encoding="utf-8"))
match = [roi for roi in rois if fold(roi["name"]) == fold(ROI_NAME)]
if len(match) != 1:
    names = ", ".join(roi["name"] for roi in rois)
    raise KeyError(f"{ROI_NAME!r} not found in {ROIS_PATH.name}. Names: {names}")
roi = match[0]
box = roi["bbox"]
AOI = (box["west"], box["south"], box["east"], box["north"])
SHOW_TIDE = fold(roi["name"]) == fold("Belém")
site_slug = "_".join(
    part for part in "".join(c if c.isalnum() else " " for c in fold(roi["name"])).split()
)
OUT_STEM = f"belem_glwd_tidal_map_{site_slug}"
OUT_PNG = HERE / f"{OUT_STEM}.png"
print(f"{roi['name']} — {roi['region']}")
print(f"bbox {AOI[0]}, {AOI[1]} : {AOI[2]}, {AOI[3]}")

# Third panel: the study bbox plus a small margin.
ZOOM_PAD_DEG = 0.02
ZOOM = (
    AOI[0] - ZOOM_PAD_DEG,
    AOI[1] - ZOOM_PAD_DEG,
    AOI[2] + ZOOM_PAD_DEG,
    AOI[3] + ZOOM_PAD_DEG,
)
# Belém only: AOI plus the Ribamar gauge (São Luís island).
REGIONAL = (-50.0, -3.2, -43.4, -0.9)
LOCATOR_PAD_DEG = 12.0
LOCATOR = (
    max(-160.0, AOI[0] - LOCATOR_PAD_DEG),
    max(-56.0, AOI[1] - LOCATOR_PAD_DEG),
    min(180.0, AOI[2] + LOCATOR_PAD_DEG),
    min(72.0, AOI[3] + LOCATOR_PAD_DEG),
)

# SiMCosta / IOC station riba. The CSV `sensor` column is `rad`.
STATION = {"name": "São José do Ribamar", "lon": -44.05167, "lat": -2.56333}
# Approximate urban centres, not survey points.
CITIES = [
    {"name": "Belém", "lon": -48.5044, "lat": -1.4558},
    {"name": "Ananindeua", "lon": -48.3722, "lat": -1.3656},
    {"name": "São Luís", "lon": -44.3028, "lat": -2.5297},
]

# Display colours. The CSV has names only. Dryland and nodata are not drawn.
GLWD_RGB = {
    1: (30, 90, 190),
    2: (70, 120, 170),
    3: (40, 140, 180),
    4: (15, 60, 150),
    5: (70, 160, 220),
    6: (120, 180, 220),
    7: (150, 210, 230),
    8: (20, 90, 60),
    9: (110, 150, 90),
    10: (0, 110, 70),
    11: (130, 200, 140),
    12: (210, 110, 30),
    13: (240, 190, 70),
    14: (130, 80, 40),
    15: (190, 160, 110),
    16: (0, 80, 50),
    17: (80, 160, 100),
    18: (90, 130, 80),
    19: (170, 190, 130),
    20: (170, 80, 100),
    21: (210, 160, 160),
    22: (90, 60, 40),
    23: (160, 120, 80),
    24: (120, 70, 40),
    25: (180, 140, 90),
    26: (90, 50, 20),
    27: (170, 120, 70),
    28: (20, 100, 40),
    29: (180, 140, 180),
    # Silt rather than the peat/forest browns.
    30: (214, 176, 92),
    # Coastal marsh: teal, not the peach used before.
    31: (48, 150, 156),
    32: (200, 180, 140),
    33: (210, 200, 80),
}
# Drop classes that cover less than this share of wetland pixels in the AOI.
MIN_WETLAND_SHARE = 0.01

HALF = 20037508.342789244
OSM = "https://tile.openstreetmap.org/{z}/{x}/{y}.png"
TILE_CACHE = HERE / "osm_cache"
GLWD_TIF = GLWD_ROOT / "GLWD_v2_0_combined_classes" / "GLWD_v2_0_main_class.tif"
GLWD_LEGEND_CSV = GLWD_ROOT / "GLWD_Legend_v2_0.csv"

with GLWD_LEGEND_CSV.open(newline="", encoding="utf-8") as handle:
    GLWD_NAME = {int(row["GLWD_ID"]): row["Class_name"] for row in csv.DictReader(handle)}


def lonlat_to_mercator(lon, lat):
    lon = np.asarray(lon, dtype=np.float64)
    lat = np.asarray(lat, dtype=np.float64)
    x = lon / 180.0 * HALF
    y = np.log(np.tan(np.pi / 4.0 + np.radians(lat) / 2.0)) * HALF / np.pi
    return x, y


def mercator_to_lonlat(x, y):
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    lon = x / HALF * 180.0
    lat = np.degrees(2.0 * np.arctan(np.exp(y * np.pi / HALF)) - np.pi / 2.0)
    return lon, lat


def haversine_km(lon1, lat1, lon2, lat2):
    radius = 6371.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp = math.radians(lat2 - lat1)
    dl = math.radians(lon2 - lon1)
    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * radius * math.asin(math.sqrt(a))


def osm_mosaic(west, south, east, north, zoom):
    """RGB mosaic and its EPSG:3857 extent (left, right, bottom, top)."""
    n = 2**zoom

    def tile_xy(lon, lat):
        x = (lon + 180.0) / 360.0 * n
        lat_r = math.radians(lat)
        y = (1.0 - math.asinh(math.tan(lat_r)) / math.pi) / 2.0 * n
        return x, y

    x0, y_north = tile_xy(west, north)
    x1, y_south = tile_xy(east, south)
    c0, c1 = max(int(math.floor(x0)), 0), min(int(math.floor(x1)), n - 1)
    r0, r1 = max(int(math.floor(y_north)), 0), min(int(math.floor(y_south)), n - 1)
    mosaic = np.full(((r1 - r0 + 1) * 256, (c1 - c0 + 1) * 256, 3), 230, dtype=np.uint8)
    TILE_CACHE.mkdir(parents=True, exist_ok=True)
    for row in range(r0, r1 + 1):
        for col in range(c0, c1 + 1):
            cache = TILE_CACHE / f"{zoom}_{col}_{row}.png"
            if not cache.exists():
                url = OSM.format(z=zoom, x=col, y=row)
                req = urllib.request.Request(url, headers={"User-Agent": "biomass-hackathon-map/1.0"})
                cache.write_bytes(urllib.request.urlopen(req, timeout=60).read())
            # OSM tiles are paletted PNGs. Rasterio returns the index, not RGB.
            rgb = np.asarray(Image.open(cache).convert("RGB"))
            i0, j0 = (row - r0) * 256, (col - c0) * 256
            mosaic[i0 : i0 + 256, j0 : j0 + 256] = rgb

    def corner(col, row):
        lon = col / n * 360.0 - 180.0
        lat = math.degrees(math.atan(math.sinh(math.pi * (1 - 2 * row / n))))
        x, y = lonlat_to_mercator(lon, lat)
        return float(x), float(y)

    left, top = corner(c0, r0)
    right, bottom = corner(c1 + 1, r1 + 1)
    return mosaic, (left, right, bottom, top)


def class_rgba(codes, keep):
    """Wetland classes in `keep` only. Everything else stays transparent."""
    rgba = np.zeros((*codes.shape, 4), dtype=np.float32)
    for code in np.unique(codes):
        code = int(code)
        if code not in keep:
            continue
        rgba[codes == code, :3] = np.array(GLWD_RGB[code]) / 255.0
        rgba[codes == code, 3] = 0.82
    return rgba


def read_glwd(west, south, east, north):
    """Window of the global class grid. CRS is read from the file (EPSG:4326)."""
    with rasterio.open(GLWD_TIF) as ds:
        if ds.crs != CRS.from_epsg(4326):
            raise RuntimeError(f"expected EPSG:4326, got {ds.crs}")
        window = window_from_bounds(west, south, east, north, ds.transform).round_offsets().round_lengths()
        data = ds.read(1, window=window, boundless=True, fill_value=int(ds.nodata))
        transform = ds.window_transform(window)
        return data, transform, ds.crs, float(ds.nodata)


def glwd_on_view(west, south, east, north, keep):
    """Nearest-neighbour sample onto an EPSG:3857 grid at the source pixel size."""
    src, src_transform, src_crs, nodata = read_glwd(west, south, east, north)
    pixel = abs(src_transform.a)
    width = max(1, int(round((east - west) / pixel)))
    height = max(1, int(round((north - south) / pixel)))
    x0, y0 = lonlat_to_mercator(west, south)
    x1, y1 = lonlat_to_mercator(east, north)
    dst_transform = transform_from_bounds(float(x0), float(y0), float(x1), float(y1), width, height)
    dst = np.full((height, width), nodata, dtype=np.uint8)
    reproject(
        source=src,
        destination=dst,
        src_transform=src_transform,
        src_crs=src_crs,
        src_nodata=nodata,
        dst_transform=dst_transform,
        dst_crs=CRS.from_epsg(3857),
        dst_nodata=nodata,
        resampling=Resampling.nearest,
    )
    return class_rgba(dst, keep)


def limits(west, south, east, north):
    x0, y0 = lonlat_to_mercator(west, south)
    x1, y1 = lonlat_to_mercator(east, north)
    return float(x0), float(x1), float(y0), float(y1)


def style_axes(ax, west, south, east, north, lon_step, lat_step):
    x0, x1, y0, y1 = limits(west, south, east, north)
    ax.set_xlim(x0, x1)
    ax.set_ylim(y0, y1)
    ax.set_aspect("equal")
    lons = np.arange(math.ceil(west / lon_step) * lon_step, east, lon_step)
    lats = np.arange(math.ceil(south / lat_step) * lat_step, north, lat_step)
    ax.xaxis.set_major_locator(FixedLocator(lonlat_to_mercator(lons, np.zeros_like(lons))[0]))
    ax.yaxis.set_major_locator(FixedLocator(lonlat_to_mercator(np.zeros_like(lats), lats)[1]))
    decimals = 0 if max(lon_step, lat_step) >= 1 else 1 if max(lon_step, lat_step) >= 0.1 else 2
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _p, n=decimals: f"{float(mercator_to_lonlat(v, 0)[0]):.{n}f}°"))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _p, n=decimals: f"{float(mercator_to_lonlat(0, v)[1]):.{n}f}°"))
    ax.tick_params(labelsize=8)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")


def add_box(ax, west, south, east, north, **kwargs):
    xs, ys = lonlat_to_mercator([west, east, east, west], [south, south, north, north])
    ax.add_patch(MplPolygon(np.column_stack([xs, ys]), closed=True, fill=False, **kwargs))


def add_marker(ax, lon, lat, text, dx, dy, **plot_kw):
    x, y = lonlat_to_mercator(lon, lat)
    ax.plot(float(x), float(y), **plot_kw)
    ax.annotate(
        text,
        (float(x), float(y)),
        textcoords="offset points",
        xytext=(dx, dy),
        fontsize=8,
        bbox={"boxstyle": "round,pad=0.15", "fc": "white", "ec": "none", "alpha": 0.85},
    )


def zoom_level(west, south, east, north, tiles=6):
    span = max(east - west, north - south, 0.05)
    level = math.ceil(math.log2(360.0 * tiles / span))
    return int(min(13, max(4, level)))


def tick_step(span):
    for step in (20, 10, 5, 2, 1, 0.5, 0.2, 0.1, 0.05, 0.02):
        if span / step >= 2:
            return step
    return span / 4


aoi_classes, _, _, _ = read_glwd(*AOI)
counts = {int(code): int(n) for code, n in zip(*np.unique(aoi_classes, return_counts=True))}
wetland_px = sum(n for code, n in counts.items() if code in GLWD_RGB)
shown = []
if wetland_px:
    shown = [
        code
        for code, n in sorted(counts.items())
        if code in GLWD_RGB and n / wetland_px >= MIN_WETLAND_SHARE
    ]
for code, count in sorted(counts.items()):
    label = GLWD_NAME.get(code, "nodata")
    flag = "" if code in shown or code not in GLWD_RGB else "  (hidden, <1% of wetland px)"
    print(f"{code:3d}  {label:48s}  {count:6d} px{flag}")

aoi_lon, aoi_lat = (AOI[0] + AOI[2]) / 2, (AOI[1] + AOI[3]) / 2
if SHOW_TIDE:
    km_aoi = haversine_km(STATION["lon"], STATION["lat"], aoi_lon, aoi_lat)
    print(f"Ribamar tide gauge is {km_aoi:.0f} km from the AOI centre")

base_loc, ext_loc = osm_mosaic(*LOCATOR, zoom=zoom_level(*LOCATOR, tiles=4))
base_zoom, ext_zoom = osm_mosaic(*ZOOM, zoom=zoom_level(*ZOOM, tiles=6))
wet_rgba = glwd_on_view(*ZOOM, shown)
if SHOW_TIDE:
    base_reg, ext_reg = osm_mosaic(*REGIONAL, zoom=7)

fig = plt.figure(figsize=(14.5 if SHOW_TIDE else 11.5, 9.2), layout="constrained")
width = [0.85, 2.15, 2.15] if SHOW_TIDE else [0.9, 2.2]
grid = fig.add_gridspec(2, len(width), height_ratios=[1, 0.55], width_ratios=width)
axes = [fig.add_subplot(grid[0, i]) for i in range(len(width))]

axes[0].imshow(base_loc, extent=ext_loc, origin="upper", interpolation="bilinear")
style_axes(axes[0], *LOCATOR, lon_step=tick_step(LOCATOR[2] - LOCATOR[0]), lat_step=tick_step(LOCATOR[3] - LOCATOR[1]))
add_box(axes[0], *AOI, edgecolor="#d62728", linewidth=1.4, zorder=3)
x_roi, y_roi = lonlat_to_mercator(aoi_lon, aoi_lat)
axes[0].plot(float(x_roi), float(y_roi), marker="o", color="#d62728", markersize=6, zorder=4)
axes[0].set_title(roi["region"], fontsize=10)
axes[0].set_xlabel("")
axes[0].set_ylabel("")
axes[0].set_xticks([])
axes[0].set_yticks([])

zoom_ax = axes[2] if SHOW_TIDE else axes[1]
if SHOW_TIDE:
    axes[1].imshow(base_reg, extent=ext_reg, origin="upper", interpolation="bilinear")
    style_axes(axes[1], *REGIONAL, lon_step=1.0, lat_step=0.5)
    add_box(axes[1], *AOI, edgecolor="black", linewidth=1.4, zorder=3)
    add_marker(
        axes[1], STATION["lon"], STATION["lat"], "Ribamar tide gauge", 8, 6,
        marker="^", color="#d62728", markersize=9, zorder=5,
    )
    axes[1].set_title(f"Tide gauge is {km_aoi:.0f} km from the AOI centre")

zoom_ax.imshow(base_zoom, extent=ext_zoom, origin="upper", interpolation="bilinear")
x0, x1, y0, y1 = limits(*ZOOM)
zoom_ax.imshow(
    wet_rgba,
    extent=(x0, x1, y0, y1),
    origin="upper",
    interpolation="nearest",
    zorder=2,
)
style_axes(
    zoom_ax, *ZOOM,
    lon_step=tick_step(ZOOM[2] - ZOOM[0]),
    lat_step=tick_step(ZOOM[3] - ZOOM[1]),
)
add_box(zoom_ax, *AOI, edgecolor="black", linewidth=1.4, zorder=3)
zoom_ax.set_title(f"{roi['name']} — GLWD v2")

handles = [plt.Line2D([0], [0], color="black", linewidth=1.4, label="Study bbox")]
if SHOW_TIDE:
    handles.append(
        plt.Line2D([0], [0], marker="^", color="none", markerfacecolor="#d62728", markersize=8, label="Tide gauge")
    )
handles += [
    Patch(facecolor=np.array(GLWD_RGB[code]) / 255, edgecolor="none", label=GLWD_NAME[code])
    for code in shown
]
legend_ax = fig.add_subplot(grid[1, :])
legend_ax.axis("off")
legend_ax.legend(
    handles=handles,
    loc="upper center",
    ncol=2,
    frameon=False,
    fontsize=13,
    markerscale=1.6,
    handlelength=1.6,
    labelspacing=0.6,
)
fig.text(0.01, 0.01, "Basemap © OpenStreetMap contributors", fontsize=7, color="0.35")
fig.savefig(OUT_PNG, dpi=160, bbox_inches="tight")
print(f"wrote {OUT_PNG}")
plt.show()


In [ ]:
import folium
from rasterio.transform import array_bounds

glw_aoi, glwd_transform, _, _ = read_glwd(*AOI)
rga = class_rgba(glw_aoi, shown)
image = (np.clip(rga, 0, 1) * 255).astype(np.uint8)
height, width = glw_aoi.shape
west, south, east, north = array_bounds(height, width, glwd_transform)

local = folium.Map(tiles="OpenStreetMap", control_scale=True)
local.fit_bounds([[ZOOM[1], ZOOM[0]], [ZOOM[3], ZOOM[2]]])
folium.raster_layers.ImageOverlay(
    image=image,
    bounds=[[south, west], [north, east]],
    opacity=0.85,
    mercator_project=True,
    name="GLWD v2",
).add_to(local)
folium.Rectangle(
    bounds=[[AOI[1], AOI[0]], [AOI[3], AOI[2]]],
    color="black",
    weight=2,
    fill=False,
    tooltip="Study bbox",
).add_to(local)
for city in CITIES:
    if not (ZOOM[0] <= city["lon"] <= ZOOM[2] and ZOOM[1] <= city["lat"] <= ZOOM[3]):
        continue
    folium.CircleMarker(
        [city["lat"], city["lon"]],
        radius=5,
        color="black",
        fill=True,
        fill_opacity=1,
        tooltip=city["name"],
    ).add_to(local)
folium.LayerControl().add_to(local)
local.save(HERE / f"{OUT_STEM}.html")

map_outputs = [local]
if SHOW_TIDE:
    region = folium.Map(tiles="OpenStreetMap", control_scale=True)
    region.fit_bounds([[REGIONAL[1], REGIONAL[0]], [REGIONAL[3], REGIONAL[2]]])
    folium.Rectangle(
        bounds=[[AOI[1], AOI[0]], [AOI[3], AOI[2]]],
        color="black",
        weight=2,
        fill=False,
        tooltip="Study bbox",
    ).add_to(region)
    for city in CITIES:
        folium.CircleMarker(
            [city["lat"], city["lon"]],
            radius=4,
            color="black",
            fill=True,
            fill_opacity=1,
            tooltip=city["name"],
        ).add_to(region)
    folium.Marker(
        [STATION["lat"], STATION["lon"]],
        tooltip=f"{STATION['name']} tide gauge ({km_aoi:.0f} km from AOI centre)",
        icon=folium.Icon(color="red", icon="info-sign"),
    ).add_to(region)
    region.save(HERE / f"{OUT_STEM}_tide.html")
    map_outputs.append(region)

from IPython.display import display

for item in map_outputs:
    display(item)
